# Cube state recognition

YOLO segmentation of stickers, then color assignment A, B, B+, C, C+ evaluated against the scrambles.

## Setup

In [ ]:
!pip install -q ultralytics "dvc==3.67.1" loguru python-dotenv typer

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
REPO = "/content/puzzle-vision"
DVC_REMOTE = "/content/drive/MyDrive/puzzle-vision-dvc"  # folder of the gdrive remote

In [ ]:
!test -d {REPO} || git clone -q https://github.com/kwyrzyk/puzzle-vision.git {REPO}
%cd {REPO}/research
# mounted Drive as a local remote: same layout as gdrive, no OAuth secret needed
!dvc remote add -f --local -d drive {DVC_REMOTE}
!dvc pull -q data/raw/images data/raw/annotations
!dvc repro

In [ ]:
import sys

sys.path.insert(0, f"{REPO}/research")  # pip install needs Python 3.14

from collections import Counter
import csv
from pathlib import Path
from typing import NamedTuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.vq import kmeans2
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from ultralytics import YOLO

from puzzle_vision.annotations import PHOTO_TOP, ImageAnnotation, grid_order
from puzzle_vision.config import PROCESSED_DATA_DIR, RAW_METADATA_DIR
from puzzle_vision.cube import CENTER_IDX, COLOR, FACELETS_PER_FACE, CubeState, Face, face_view
from puzzle_vision.predictions import deduplicate, from_result, to_annotation

In [ ]:
DATASET = PROCESSED_DATA_DIR / "yolo_seg"
RUNS = Path("/content/drive/MyDrive/rubik_runs")
RUN = "seg_v2"
SPLIT = "val"
CONF = 0.25
IMGSZ = 640

## Training

In [ ]:
model = YOLO("yolo11n-seg.pt")
model.train(
    data=str(DATASET / "data.yaml"),
    epochs=100,
    imgsz=IMGSZ,
    batch=16,
    patience=20,
    project=str(RUNS),  # Drive, Colab storage is lost after the session
    name=RUN,
)

In [ ]:
model = YOLO(str(RUNS / RUN / "weights" / "best.pt"))
model.val(data=str(DATASET / "data.yaml"), split=SPLIT, imgsz=IMGSZ)

## Predictions

In [ ]:
FACE_OF_COLOR = {color: face for face, color in COLOR.items()}


class Cell(NamedTuple):
    yolo: Face  # color of the YOLO class
    rgb: np.ndarray  # median inside the mask


def photo(path, size):
    cube, number, index, color = path.stem.split("_")  # cube_0001_1_white
    return ImageAnnotation(
        f"{cube}_{number}", f"{index}_{color}{path.suffix}", size, FACE_OF_COLOR[color], [], []
    )


def median_rgb(image_bgr, polygon):
    mask = np.zeros(image_bgr.shape[:2], np.uint8)
    cv2.fillPoly(mask, [np.int32(polygon)], 255)
    eroded = cv2.erode(mask, np.ones((9, 9), np.uint8))  # without edges and shadows
    if eroded.any():
        mask = eroded
    return np.median(image_bgr[mask > 0], axis=0)[::-1]

In [ ]:
cubes = {}  # cube_id -> {(face, index): Cell}
detected = {}
for r in model.predict(
    str(DATASET / "images" / SPLIT), conf=CONF, imgsz=IMGSZ, stream=True, verbose=False
):
    image = photo(Path(r.path), r.orig_shape[::-1])
    predicted = to_annotation(image, deduplicate(from_result(r)))
    detected[Path(r.path).stem] = len(predicted.stickers)
    cells = cubes.setdefault(image.cube_id, {})
    if len(predicted.stickers) != FACELETS_PER_FACE:
        continue  # grid unknown, the face stays unassigned
    order = grid_order([sticker.center for sticker in predicted.stickers])
    for index, i in zip(face_view(image.face, PHOTO_TOP[image.face]), order, strict=True):
        sticker = predicted.stickers[i]
        cells[image.face, index] = Cell(sticker.color, median_rgb(r.orig_img, sticker.polygon))

print(f"{len(cubes)} cubes")
pd.Series(detected).value_counts().rename("stickers after deduplication")

In [ ]:
CUBE = min(cubes)

paths = sorted(str(p) for p in (DATASET / "images" / SPLIT).glob(f"{CUBE}_*"))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, r in zip(
    axes.flat, model.predict(paths, conf=CONF, imgsz=IMGSZ, verbose=False), strict=True
):
    ax.imshow(r.plot()[:, :, ::-1])
    ax.set_title(Path(r.path).stem)
    ax.axis("off")
plt.tight_layout()

## Color assignment

- A: YOLO class
- B: nearest center color in RGB, at most 9 per color; center colors from file names
- C: as B, center colors from k-means
- B+, C+: a color is set only if `CubeState.defects()` finds nothing new

In [ ]:
CANONICAL_RGB = {
    Face.U: (255, 255, 255),
    Face.R: (220, 30, 40),
    Face.F: (0, 160, 0),
    Face.D: (230, 210, 0),
    Face.L: (255, 130, 0),
    Face.B: (0, 60, 200),
}
CANONICAL = np.array([CANONICAL_RGB[face] for face in Face], np.float32)


def approach_a(cells):
    state = CubeState()
    for (face, index), cell in cells.items():
        state.set_color(face, index, cell.yolo)
    return state


def file_centers(cells):
    return {face: face for face in Face if (face, CENTER_IDX) in cells}


def kmeans_centers(cells, seed=0):
    # clusters of all stickers matched to the centers, then to the canonical colors
    centers = [face for face in Face if (face, CENTER_IDX) in cells]
    rgb = np.array([cell.rgb for cell in cells.values()], np.float32)
    clusters, _ = kmeans2(rgb, len(Face), minit="++", seed=seed)
    center_rgb = np.array([cells[face, CENTER_IDX].rgb for face in centers])
    _, cluster = linear_sum_assignment(np.linalg.norm(center_rgb[:, None] - clusters, axis=2))
    _, color = linear_sum_assignment(
        np.linalg.norm(clusters[cluster][:, None] - CANONICAL, axis=2)
    )
    return {face: Face(c) for face, c in zip(centers, color, strict=True)}


def assign(cells, center_colors, validate):
    state = CubeState()
    for face, color in center_colors.items():
        state.set_color(face, CENTER_IDX, color)
    refs = {color: cells[face, CENTER_IDX].rgb for face, color in center_colors.items()}
    pairs = sorted(
        (np.linalg.norm(cell.rgb - ref), place, color)
        for place, cell in cells.items()
        if place[1] != CENTER_IDX
        for color, ref in refs.items()
    )
    counts = Counter(center_colors.values())
    known = set(state.defects())  # e.g. CENTERS for wrong k-means centers
    for _, (face, index), color in pairs:
        if state.color(face, index) is not None:
            continue
        trial = CubeState.from_facelets(str(state))
        trial.set_color(face, index, color)
        if validate and set(trial.defects()) - known:
            continue
        if not validate and counts[color] == FACELETS_PER_FACE:
            continue
        state = trial
        counts[color] += 1
    return state

In [ ]:
APPROACHES = {
    "A": approach_a,
    "B": lambda cells: assign(cells, file_centers(cells), validate=False),
    "B+": lambda cells: assign(cells, file_centers(cells), validate=True),
    "C": lambda cells: assign(cells, kmeans_centers(cells), validate=False),
    "C+": lambda cells: assign(cells, kmeans_centers(cells), validate=True),
}
states = {
    name: {cube_id: approach(cells) for cube_id, cells in cubes.items()}
    for name, approach in APPROACHES.items()
}

## Evaluation

In [ ]:
with (RAW_METADATA_DIR / "cubes.csv").open(newline="") as f:
    truth = {
        row["cube_id"]: CubeState.from_scramble(row["scramble"])
        for row in csv.DictReader(f)
        if row["cube_id"] in cubes
    }

BACKGROUND = "background"  # facelet without color
LABELS = [*COLOR.values(), BACKGROUND]


def facelet_colors(state):
    colors = (state.color(face, i) for face in Face for i in range(FACELETS_PER_FACE))
    return [BACKGROUND if color is None else COLOR[color] for color in colors]


def correct(state, cube_id):
    return sum(
        p == t for p, t in zip(facelet_colors(state), facelet_colors(truth[cube_id]), strict=True)
    )


def accuracy(by_cube):
    return sum(correct(state, cube_id) for cube_id, state in by_cube.items()) / (
        len(by_cube) * len(facelet_colors(CubeState()))
    )


assert accuracy(truth) == 1
assert accuracy({cube_id: CubeState() for cube_id in truth}) == 0

In [ ]:
pd.DataFrame(
    {
        name: {
            "facelet accuracy": accuracy(by_cube),
            "cubes correct": sum(state == truth[cube_id] for cube_id, state in by_cube.items()),
            "cubes solvable": sum(
                state.is_complete() and not state.defects() for state in by_cube.values()
            ),
        }
        for name, by_cube in states.items()
    }
).T.assign(cubes=len(truth))

In [ ]:
fig, axes = plt.subplots(1, len(states), figsize=(6 * len(states), 5))
for ax, (name, by_cube) in zip(axes, states.items(), strict=True):
    y_true = [c for cube_id in by_cube for c in facelet_colors(truth[cube_id])]
    y_pred = [c for state in by_cube.values() for c in facelet_colors(state)]
    ConfusionMatrixDisplay(
        confusion_matrix(y_true, y_pred, labels=LABELS), display_labels=LABELS
    ).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
    ax.set_title(name)
plt.tight_layout()

In [ ]:
pd.DataFrame(
    {
        name: {cube_id: correct(state, cube_id) for cube_id, state in by_cube.items()}
        for name, by_cube in states.items()
    }
).rename_axis("correct facelets of 54")

In [ ]:
NET = {
    Face.U: (0, 1),
    Face.L: (1, 0),
    Face.F: (1, 1),
    Face.R: (1, 2),
    Face.B: (1, 3),
    Face.D: (2, 1),
}


def show_net(ax, state, expected, title):
    for face, (row, col) in NET.items():
        for i in range(FACELETS_PER_FACE):
            x, y = col * 3 + i % 3, -(row * 3 + i // 3)
            color = state.color(face, i)
            fill = "lightgray" if color is None else COLOR[color]
            ax.add_patch(plt.Rectangle((x, y), 1, 1, facecolor=fill, edgecolor="black"))
            if color != expected.color(face, i):
                ax.text(x + 0.5, y + 0.5, "x", ha="center", va="center", fontsize=14)
    ax.set_xlim(0, 12)
    ax.set_ylim(-8, 1)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title)


fig, axes = plt.subplots(2, 3, figsize=(18, 9))
show_net(axes.flat[0], truth[CUBE], truth[CUBE], f"{CUBE} scramble")
for ax, (name, by_cube) in zip(axes.flat[1:], states.items(), strict=True):
    show_net(ax, by_cube[CUBE], truth[CUBE], f"{name}: {correct(by_cube[CUBE], CUBE)}/54")
plt.tight_layout()